# 00 — Python and tensors for ACORN

**Solution notebook · 45–60 minutes · CPU only**

This is a focused Python survival kit, not a complete Python course. Every idea appears because you will meet it in ACORN model code. By the end, you will be able to follow how named hit features become one tensor and then one row of inputs per graph edge.

## How to use this notebook

Run a cell with **Shift+Enter**. Read the cells in order: later cells reuse earlier variables. The student notebook contains six exercises; this version contains their completed answers. `assert` statements are small automatic checks. No output from an assertion means the check passed.

In [ ]:
import math
import traceback

import torch

torch.manual_seed(7)
print("PyTorch version:", torch.__version__)
print("CUDA needed:", False)

## 1. Names and values

A variable gives a value a name. Python determines the value's type for us. ACORN configurations and models use strings for feature names, integers for layer sizes, floating-point numbers for cuts, and booleans for switches.

In [ ]:
detector_name = "toy detector"  # str: text
number_of_hits = 4              # int: a whole number
edge_cut = 0.5                  # float: a decimal number
use_attention = False           # bool: True or False

print(detector_name, number_of_hits, edge_cut, use_attention)
print(type(number_of_hits), type(edge_cut))

In [ ]:
# Exercise 1 — create three variables with the requested values.
event_name = "tiny_event"
hit_count = 4
is_simulated = True

assert event_name == "tiny_event"
assert hit_count == 4
assert is_simulated is True

## 2. Lists, dictionaries, loops, and comprehensions

A **list** is an ordered collection. A **dictionary** maps keys to values. In ACORN, `node_features` is commonly a list, while loaded YAML configuration behaves like a dictionary. Python starts indexing at zero.

In [ ]:
features = ["r", "phi", "z"]
config = {"node_features": features, "hidden": 8}

print("first feature:", features[0])
print("hidden size:", config["hidden"])

for feature in features:
    print("feature:", feature)

upper_names = [feature.upper() for feature in features]
print(upper_names)

In [ ]:
# Exercise 2 — build an ACORN-like configuration.
node_features = ["r", "phi", "z"]
model_config = {"node_features": node_features, "hidden": 16}
feature_labels = [name.upper() for name in node_features]

assert model_config["node_features"][1] == "phi"
assert model_config["hidden"] == 16
assert feature_labels == ["R", "PHI", "Z"]

## 3. Functions, return values, and imports

A function packages an operation under a name. Parameters are its inputs; `return` sends a result back to the caller. Printing shows a value to a human, but returning makes it available to other code. `import` makes names from another module available—`torch` and Python's built-in `math` module are examples.

In [ ]:
def cylindrical_distance(r, phi):
    x = r * math.cos(phi)
    y = r * math.sin(phi)
    return math.sqrt(x**2 + y**2)

distance = cylindrical_distance(3.0, 0.4)
print(distance)
assert math.isclose(distance, 3.0)

In [ ]:
# Exercise 3 — return a tensor's name, shape, and dtype.
def describe_tensor(name, tensor):
    return name, tuple(tensor.shape), tensor.dtype

example = torch.tensor([1.0, 2.0, 3.0])
description = describe_tensor("example", example)
print(description)
assert description == ("example", (3,), torch.float32)

## 4. Tensors: shape, dtype, indexing, and masks

A tensor is a rectangular collection of values. `shape` describes the size of each dimension and `dtype` describes how values are stored. Here each one-dimensional tensor contains one value per hit, so its shape is `[number of hits] = [4]`.

In [ ]:
r = torch.tensor([1.0, 2.0, 3.0, 4.0])
phi = torch.tensor([0.1, -0.2, 0.3, -0.4])
z = torch.tensor([-3.0, -1.0, 1.0, 3.0])

print("r:", r)
print("shape:", r.shape, "dtype:", r.dtype)  # [4]
print("first r:", r[0], "last r:", r[-1])
assert r.shape == (4,)

In [ ]:
# Exercise 4 — select z only for hits with r greater than 2.
outer_mask = r > 2.0
outer_z = z[outer_mask]

print("mask:", outer_mask)       # [4], bool
print("selected z:", outer_z)   # [2], float
assert outer_mask.dtype == torch.bool
assert torch.equal(outer_mask, torch.tensor([False, False, True, True]))
assert torch.equal(outer_z, torch.tensor([1.0, 3.0]))

## 5. Combining tensors for graph edges

`torch.stack` creates a new dimension. Stacking three `[4]` feature tensors along the last dimension produces node features with shape `[4, 3]`: four hits, three features per hit.

`edge_index` has shape `[2, 5]`. Row 0 contains the source-hit index and row 1 contains the destination-hit index for each of five candidate edges. `torch.cat` joins existing dimensions. Joining two `[5, 3]` endpoint tensors produces `[5, 6]`.

In [ ]:
edge_index = torch.tensor([
    [0, 1, 1, 2, 3],  # source hits
    [1, 0, 2, 3, 2],  # destination hits
])
start, end = edge_index
print("edge_index:", edge_index.shape)  # [2, 5]
print("start:", start.shape)            # [5]

In [ ]:
# Exercise 5 — stack node features, gather endpoints, then concatenate.
x = torch.stack([r, phi, z], dim=-1)
start_features = x[start]
end_features = x[end]
edge_inputs = torch.cat([start_features, end_features], dim=-1)

print("nodes:", x.shape)                   # [4, 3]
print("source features:", start_features.shape)  # [5, 3]
print("edge inputs:", edge_inputs.shape)  # [5, 6]
assert x.shape == (4, 3)
assert edge_inputs.shape == (5, 6)
assert torch.equal(edge_inputs[0], torch.cat([x[0], x[1]]))

## 6. Classes, inheritance, and `forward()`

A class describes objects that keep data and provide methods. `self` means the current object. `__init__` runs when an object is created. A subclass inherits behaviour from a parent class; `super().__init__()` initializes that parent.

PyTorch models inherit from `torch.nn.Module`. Layers are created in `__init__`; tensor computation belongs in `forward`. Calling `model(tensor)` asks `nn.Module` to invoke `forward` and preserves PyTorch's hooks—prefer it over calling `model.forward(tensor)` yourself.

In [ ]:
class FeatureScaler(torch.nn.Module):
    def __init__(self, factor):
        super().__init__()
        self.factor = factor

    def forward(self, tensor):
        return tensor * self.factor


scaler = FeatureScaler(factor=10.0)
print(scaler(r))

In [ ]:
# Exercise 6 — subclass FeatureScaler and remember a feature name.
class NamedFeatureScaler(FeatureScaler):
    def __init__(self, factor, feature_name):
        super().__init__(factor)
        self.feature_name = feature_name


named_scaler = NamedFeatureScaler(2.0, "r")
scaled_r = named_scaler(r)
assert named_scaler.feature_name == "r"
assert named_scaler.factor == 2.0
assert torch.equal(scaled_r, torch.tensor([2.0, 4.0, 6.0, 8.0]))

## 7. Changing an object versus returning a new object

Some operations mutate an existing object; others return a new one. This matters when the same event or configuration is used in several places. Lists' `append` method mutates. Tensor multiplication returns a new tensor. Use `clone()` before an in-place tensor update when the original must remain unchanged.

In [ ]:
original_features = ["r", "phi"]
same_list = original_features
same_list.append("z")
print(original_features)  # changed through the other name

original_r = r
new_r = original_r * 2
assert torch.equal(original_r, r)
assert not torch.equal(original_r, new_r)

## 8. Reading a traceback

A traceback tells you where an error travelled through the program. Start at the final line: it gives the exception type and message. Then move upward to the first line that points into code you control. The example catches the exception so this notebook can continue, but prints the useful end of its traceback.

In [ ]:
try:
    config["number_of_layers"]
except KeyError:
    final_lines = traceback.format_exc().strip().splitlines()[-3:]
    print("\n".join(final_lines))

print("Read last: KeyError means the requested dictionary key is missing.")
print("Then inspect available keys:", list(config.keys()))

## 9. Put it together: read an ACORN-shaped data flow

The function below is intentionally smaller than a real model, but its operations mirror the opening of [`InteractionGNN.forward`](https://github.com/GNN4ITkTeam/CommonFramework/blob/f8b8787e269e0ba504d1bf0a555806b7f69d04e2/acorn/stages/edge_classifier/models/interaction_gnn.py) at the pinned compatibility revision. A real ACORN model is a class, stores configuration as `self.hparams`, and receives a PyG batch that permits named feature lookup.

Read the function one line at a time and predict each shape before running it. It returns raw edge inputs rather than scores; learning how layers turn these inputs into logits belongs in tutorial 03.

In [ ]:
def build_edge_inputs(batch, hparams):
    # One column per configured node feature: [N, F].
    node_tensor = torch.stack(
        [batch[name] for name in hparams["node_features"]], dim=-1
    ).float()

    # One source and destination index per edge: each is [E].
    start, end = batch["edge_index"]

    # Two endpoint feature vectors per edge: [E, 2 * F].
    return torch.cat([node_tensor[start], node_tensor[end]], dim=-1)


tiny_batch = {"r": r, "phi": phi, "z": z, "edge_index": edge_index}
tiny_hparams = {"node_features": ["r", "phi", "z"]}
model_inputs = build_edge_inputs(tiny_batch, tiny_hparams)

print("model inputs:", model_inputs.shape)  # [5 edges, 6 values]
assert model_inputs.shape == (edge_index.shape[1], 2 * len(tiny_hparams["node_features"]))
assert torch.isfinite(model_inputs).all()

## 10. Recap and next step

You can now identify:

- a list of feature names and a dictionary of hyperparameters;
- a function's inputs and return value;
- tensor shapes for hits (`[N]`), node features (`[N, F]`), graph connectivity (`[2, E]`), and paired edge inputs (`[E, 2F]`);
- boolean selection masks, stacking, concatenation, and indexed gathering;
- the roles of `self`, `__init__`, `super()`, and `forward()`; and
- the exception type and message at the bottom of a traceback.

Next, tutorial 01 will give these tensors physical meaning, place them in a graph event, and visualize true and false candidate edges.